# İHA Video Arama — Kaggle denemesi

Colab GPU kotanız bittiğinde kullanmak için: [poc/colab_pipeline_trial.ipynb](colab_pipeline_trial.ipynb)'nin
birebir aynısı, Kaggle'a uyarlanmış. Amaç aynı: **gerçek GPU'da** pipeline'ı
uçtan uca çalıştırıp proje-ozeti.md §8'in en kritik doğrulanmamış
varsayımını (embedding hızı, §8 40x varsayıyor) ölçmek.

## Colab'dan farkları (gerçek, doğrulanmış farklar)

| | Colab | Kaggle |
|---|---|---|
| Çalışma dizini | `/content` | `/kaggle/working` |
| Veri girişi | Drive/yükleme, yazılabilir | `/kaggle/input/...` **salt-okunur**, Dataset olarak yüklenir |
| GPU seçimi | Çalıştırma türü menüsü | Sağ panel → Settings → Accelerator (**elle seçmelisiniz**, varsayılan None) |
| İnternet | Varsayılan açık | Sağ panel → Settings → Internet - **açık olmalı** (git clone/pip için) |
| GPU kotası | Ayrı, tükenmiş olabilir | Ayrı bir kota — haftalık ~30 saat, oturum başına azami 12 saat |

> **Önemli:** Bu iki kota birbirinden bağımsız. Colab'da bittiği Kaggle'ı
> etkilemez, ama Kaggle'ın kendi haftalık sınırı var — kalibrasyon adımını
> (6. bölüm) atlamayın, boşa GPU saati harcamayın.

## Kaggle'a özgü, DOĞRULANMAMIŞ noktalar

Bu defter Colab'da çalıştığını bildiğimiz mantığın **yapısal olarak aynısı**
- ama Kaggle'ın temel Docker imajı (önceden kurulu torch/CUDA sürümü, paket
kümesi) Colab'dan **farklı**. Colab'da uğraştığımız spesifik hatalar
(`libcudart.so.13` vb.) burada hiç çıkmayabilir, ya da farklı bir hata
çıkabilir - ikisini de biz test edemedik. Karşılaştığınız hatayı bildirin.

## Bileşen durumu (Colab ile aynı mantık)

| Bileşen | Durum |
|---|---|
| Proxy üretimi (ffmpeg + NVDEC/NVENC) | ✅ |
| Embedding (Qwen3-VL-Embedding-2B) | ✅ **asıl ölçüm burada** |
| YOLO26 görsel alanlar | ✅ |
| Qdrant (gömülü mod) | ✅ işlevsel, ⚠️ performans ölçümü geçersiz |
| Nesne deposu (yerel dizin) | ✅ |
| Sorgu hattı + filtre gevşetme | ✅ |
| vLLM (yapısal ayrıştırma) | ⚠️ Colab'da CUDA sürüm sorunu yaşadık, Kaggle'da doğrulanmadı |
| Temporal + Kafka orkestrasyon | ❌ Docker yok |
| Ölçek testi (100K+) | ❌ oturum/disk sınırı |

**Qdrant gömülü mod gerçek motor değil** (tam arama, gerçek HNSW değil -
gecikme ölçümü geçersiz, işlevsel test geçerli) ve **T4 bf16 Tensor Core
içermiyor** (kod otomatik fp16'ya geçiyor, RTX 4060'ı temsil etmez) -
Colab defterindeki gerekçeler burada da aynen geçerli.

---

## Kullanım kuralları

- **Hücreleri sırayla çalıştırın.**
- **Ortam değişkenini `set_env(...)` ile değiştirin**, doğrudan `os.environ`
  ile değil (aşağıda tanımlı).
- **Modeller bir kez yüklenir** - 6. bölümden itibaren süreç-içi çalışır.
- **Qdrant istemcisini `close()` etmeyin** - önbellekli, gömülü mod dosya
  kilidi kullanıyor.
- Notebook'u **GitHub'dan taze açtığınızdan emin olun** - repo'yu güncellemek
  açık bir Kaggle sekmesindeki hücreleri otomatik güncellemez.

## 1. GPU + ortam kontrolü

**Önce:** sağ paneldeki **Settings → Accelerator**'dan bir GPU seçin.
**GPU T4 x2** önerilir (P100 değil) - T4'ün davranışını (bf16 yok, fp16'ya
geçiş) zaten Colab'da doğruladık, P100 (Pascal, compute 6.0) farklı/daha
eski bir mimari ve AWQ kuantize modellerle sorunlu olabilir. "x2" yazsa da
bu defter tek GPU kullanıyor (çoklu-GPU paralelliği kurulmadı) - ikinci GPU
boşta kalır.

GPU seçtikten sonra **Settings → Internet**'i açın (git clone / pip için
gerekli) ve oturumu yeniden başlatın.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!free -g | head -2
!df -h /kaggle/working | tail -1

## 2. Depo + bağımlılıklar

Kaggle'da torch zaten CUDA'lı geldiği için `requirements.txt`'i olduğu gibi
kurmuyoruz (mevcut torch'u bozabilir). Sadece eksikleri kuruyoruz.

`git clone` başarısız olursa: yukarıdaki Internet ayarını kontrol edin.

In [ ]:
import pathlib, subprocess, sys

REPO = pathlib.Path('/kaggle/working/VideoAnalysis')

# Sessiz git komutlari kullanmiyoruz: pull sessizce basarisiz olursa ESKI
# KOD calismaya devam eder ve hata cok sonra alakasiz bir yerde patlar.
if REPO.exists():
    print('Depo mevcut, uzak surumle esitleniyor...')
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO, check=True)
    print(subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=REPO,
                         capture_output=True, text=True).stdout.strip())
else:
    r = subprocess.run(['git', 'clone',
                        'https://github.com/ykyking1/VideoAnalysis.git', str(REPO)],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)

head = subprocess.run(['git', 'log', '--oneline', '-1'], cwd=REPO,
                      capture_output=True, text=True).stdout.strip()
print('\nCalisan surum:', head)

# Surum kontrolu DOSYA OKUYARAK - config'i BURADA IMPORT ETMIYORUZ (bir
# sonraki hucrede LOCAL_STORAGE_PATH ayarlanacak; simdi import edilirse
# modul bos degerlerle onbellege girer).
config_src = (REPO / 'common' / 'config.py').read_text(encoding='utf-8')
assert 'LOCAL_STORAGE_PATH' in config_src, (
    'ESKI KOD! git pull calismamis - yukaridaki "Calisan surum" satirini kontrol edin.')

%cd /kaggle/working/VideoAnalysis

# Kaggle'in CUDA'li torch'una DOKUNMUYORUZ - sadece eksik paketler.
# qwen-vl-utils>=0.0.14 kritik: eskisi SESSIZCE bozuk embedding uretiyor.
!pip install -q "transformers>=4.57" "qwen-vl-utils>=0.0.14" accelerate \
    qdrant-client ultralytics opencv-python-headless temporalio \
    pysolar shapely 2>&1 | tail -3

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print('Kod guncel, hazir. Sonraki hucrede ortam degiskenleri ayarlanacak.')

## 3. Ortam yapılandırması

Kaggle'da da Docker yok. İki servisi Docker'sız çalıştırıyoruz:

- **Qdrant** → gömülü mod (`QDRANT_LOCAL_PATH`)
- **Nesne deposu** → yerel dizin (`LOCAL_STORAGE_PATH`), MinIO'ya gerek yok

İkisi de `/kaggle/working` altında - **salt-okunur** `/kaggle/input`'ta değil
(oraya yazamayız, ama video kaynak dosyalarını oradan okuyabiliriz).

In [ ]:
import os, sys, importlib

def set_env(**kwargs):
    """Ortam degiskenini ayarlar VE common.config'i yeniden yukler.

    common/config.py env'i IMPORT ANINDA okuyor. Bir degiskeni (ornegin
    EMBEDDING_BATCH_SIZE) sonradan degistirirseniz, config zaten yuklenmis
    oldugu icin surec-ici cagrilar ESKI degeri gorur. Bu yardimci o tuzagi
    kapatiyor - notebook boyunca env degistirmek icin hep bunu kullanin,
    dogrudan os.environ'a yazmayin.

    Tum modullerimiz `from common import config` (modul referansi) kullandigi
    icin reload degerleri her yere yayilir."""
    for k, v in kwargs.items():
        os.environ[k] = str(v)
    if 'common.config' in sys.modules:
        importlib.reload(sys.modules['common.config'])

# Docker'siz calisan iki arka uc - /kaggle/working altinda (yazilabilir)
set_env(
    QDRANT_LOCAL_PATH='/kaggle/working/qdrant_data',
    LOCAL_STORAGE_PATH='/kaggle/working/storage',
    EMBEDDING_BATCH_SIZE=8,                     # T4 16GB icin baslangic
    EMBEDDING_DTYPE='auto',                     # T4 (compute 7.5) -> fp16
    CAPTION_ENABLED='false',                    # vLLM yok (bkz. 9. bolum)
)

from common import config
from common.minio_client import backend_name
assert config.LOCAL_STORAGE_PATH, 'LOCAL_STORAGE_PATH okunmadi'
print('Nesne deposu :', backend_name())
print('Qdrant       : gomulu ->', config.QDRANT_LOCAL_PATH)
print('Batch        :', config.EMBEDDING_BATCH_SIZE)

!python -m scripts.init_storage --skip-postgres

## 4. Ortam doğrulaması

`torch ... CPU-only` ya da `qwen-vl-utils < 0.0.14` görürseniz **durun** —
ikisi de çökmeden sessizce bozuyor. Sistem RAM uyarısı da (Colab'da bir kez
gerçekten yaşandı - model yüklenirken RAM'de çöktü, VRAM değil) ciddiye alın.

In [ ]:
!python -m scripts.check_env

## 5. Veri

**Kaggle'da `/kaggle/input` salt-okunurdur** - oraya yazamayız ama
kaynak videoyu oradan okuyup işleyebiliriz (kayıt/proxy adımı zaten
`/kaggle/working` altına kopyalıyor).

Video eklemenin iki yolu:

1. **Kaggle Dataset olarak** (önerilen): sağ panel → **Add Data** → videolarınızı
   (tek tek ya da zip) yükleyin. **Zip yüklerseniz Kaggle otomatik açar** -
   `/kaggle/input/<dataset-adi>/` altında düz dosyalar olarak görünür,
   elle unzip etmeyin.
2. **Doğrudan indirme**: `!wget ... -P /kaggle/working/videos/`

Aşağıdaki hücre HER İKİ yolu da tarar (`/kaggle/input/**` +
`/kaggle/working/videos/`), birleştirir.

Telemetri (`.tlog`) varsa `--telemetry` ile verin; yoksa `agl_m`/`over_sea`/
`sun_elevation` `None` kalır ve o filtreler test edilemez (`vehicle_count`
üzerinden test edin).

In [ ]:
import pathlib, shutil, os
from scripts.register_video import validate_video, InvalidVideoError
from scripts.ingest_all import natural_sort_key

VIDEO_EXTS = ('.mp4', '.mov', '.mkv', '.ts', '.m2ts', '.mts', '.avi')
WORKING_VIDEOS = pathlib.Path('/kaggle/working/videos')
WORKING_VIDEOS.mkdir(exist_ok=True)

# Hem Dataset olarak eklenmis (/kaggle/input, salt-okunur, iceride herhangi
# bir alt klasor yapisinda olabilir - rglob ile derin tariyoruz) hem de
# dogrudan /kaggle/working/videos'a konulmus dosyalari birlikte topluyoruz.
# Dogal (sayisal) sirada: alfabetik sira "video_10"u "video_2"den once
# koyuyordu, asagidaki N limitini yaniltiyordu (gercek Kaggle calistirmasinda
# bulundu - bkz. scripts/ingest_all.py docstring'i).
candidates = sorted(set(
    p for root in (pathlib.Path('/kaggle/input'), WORKING_VIDEOS)
    if root.exists()
    for p in root.rglob('*')
    if p.suffix.lower() in VIDEO_EXTS
), key=natural_sort_key)
assert candidates, ('Video bulunamadi. "Add Data" ile bir Dataset ekleyin ya da '
                    'dosyalari /kaggle/working/videos/ altina koyun.')

videos, bozuk = [], []
for p in candidates:
    try:
        info = validate_video(str(p))
        # Sıradaki bölüm (ingest_all.main_async) tek bir --dir bekliyor:
        # /kaggle/input'tan gelenleri WORKING_VIDEOS altina sembolik
        # baglantiyla topluyoruz (kopyalamiyoruz, buyuk dosyalarda disk
        # israfina yol acmasin).
        link = WORKING_VIDEOS / p.name
        if p.parent != WORKING_VIDEOS and not link.exists():
            os.symlink(p, link)
        videos.append(link if link.exists() else p)
        dur = f"{info['duration_s']:.1f}s" if info.get('duration_s') else '?'
        print(f"  OK    {p.name:40s} {info['size_bytes']/1024**2:8.1f} MB  "
              f"{dur:>8s}  {info.get('codec','?')}  ({p.parent})")
    except InvalidVideoError as e:
        bozuk.append(p)
        print(f"  BOZUK {p.name}")
        print(f"        {str(e).splitlines()[1].strip()}")

total_h = sum(validate_video(str(p)).get('duration_s') or 0 for p in videos) / 3600
free_gb = shutil.disk_usage('/kaggle/working').free / 1024**3
print(f"\n{len(videos)} saglam video (~{total_h:.2f} saat, ~{total_h*3600/config.WINDOW_S:.0f} pencere)")
print(f"Bos disk (/kaggle/working): {free_gb:.1f} GB  |  gereken ~{total_h*0.36 + total_h*0.36:.1f} GB "
      f"(kopya + proxy)")

if bozuk:
    print(f"\n!! {len(bozuk)} bozuk dosya atlanacak. En sik sebep yukleme yarim "
          f"kalmasi -\n   dosya boyutlarini kaynakla karsilastirin ve yeniden yukleyin.")
assert videos, 'Hicbir saglam video yok - yuklemeleri kontrol edin'

## 6. KALİBRASYON — tek video

**Toplu yüklemeden önce mutlaka bu.** Kaggle'ın haftalık ~30 saat GPU
kotası var - kalibrasyonu atlayıp doğrudan toplu yüklemeye geçmek riskli.
Çıktının sonundaki `Embedding suresi: ... (N.NNx gercek-zaman)` bu
denemenin en değerli sayısı: proje-ozeti.md §8 burada 40x varsayıyor ve
doğrulanmadı.

In [ ]:
import time
from scripts.register_video import register
from scripts.ingest_video import run_local

# SUREC ICI calistiriyoruz (`!python -m ...` degil): boylece model bir kez
# yuklenir ve sonraki hucrelerde tekrar tekrar yuklenmez.

first = videos[0]
vid = first.stem.replace(' ', '_')

register(vid, str(first))
await run_local(vid, f'{vid}/raw{first.suffix}', None, 'unknown',
                skip_caption=True)

### Batch ayarı

`EMBEDDING_BATCH_SIZE` throughput'un en büyük belirleyicisi. T4 16GB'de
8 → 16 → 24 deneyin; `CUDA out of memory` alırsanız bir kademe düşün.
Model fp16'da ~4,3 GB.

In [ ]:
# Batch degistirip AYNI videoyu tekrar olcun. set_env kullanmak sart -
# dogrudan os.environ'a yazmak surec-ici cagrilarda etkisiz kalir.
set_env(EMBEDDING_BATCH_SIZE=16)
print('Yeni batch:', config.EMBEDDING_BATCH_SIZE)

await run_local(vid, f'{vid}/raw{first.suffix}', None, 'unknown',
                skip_visual=True, skip_caption=True)

## 7. Toplu yükleme

Kalibrasyondan çıkan hıza göre kaç video sığdırabileceğinize karar verin.
**Oturum azami 12 saat, haftalık kota ~30 saat** - kısa tutun, gerekirse
tekrar çalıştırın (yazım idempotent, aynı pencere iki kez yazılmaz).

In [ ]:
import argparse
from scripts.ingest_all import main_async

# ONEMLI: bu hucre vLLM baslamadan ONCE calismali (Bolum 9 asagida). vLLM
# ayaktayken ingest calisirsa GPU bellegi paylasilir - en uzun videolarda
# CUDA OOM gorulmustu (T4 16GB, vLLM ~7.2GB'i sabit tutuyor). Notebook'un
# varsayilan sirasi (once ingest, sonra vLLM) bu sorunu zaten onluyor -
# sirayi degistirmeyin.
#
# `!python -m scripts.ingest_all` YERINE burada dogrudan cagiriyoruz: ayri
# bir surec, gomulu Qdrant'in ayni depoyu ACIK olan bu kernel'le paylasamadigi
# icin "AlreadyLocked" ile aninda cokuyordu (gercek Kaggle calistirmasinda
# bulundu). Ayrica idempotent: zaten ingest edilmis videolar otomatik atlanir,
# yarida kesilen bir yukleme ayni hucreyle kaldigi yerden devam eder.

N = None   # sinirlamak icin bir sayi verin (kalibrasyona gore); None = hepsi

args = argparse.Namespace(
    dir=str(WORKING_VIDEOS), limit=N, dry_run=False, force=False,
    sensor_type='unknown', skip_caption=True, skip_visual=False,
)
await main_async(args)

In [ ]:
from common.qdrant_store import get_client

# get_client() onbelleklidir - gomulu Qdrant dosya kilidi kullandigi icin
# ayni dizine ikinci istemci acmak hata verirdi. close() CAGIRMAYIN.
info = get_client().get_collection(config.QDRANT_COLLECTION)
print(f'{info.points_count} pencere yazildi')
print(f'~{info.points_count * config.WINDOW_S / 3600:.2f} saatlik video karsiligi')

## 8. YOLO26 dogruluk kontrolu (opsiyonel - SeaDroneSee manifest.json gerekir)

`vehicle_count` alani YOLO26'nin (COCO on-egitimli, aerial/maritime icin
fine-tune EDILMEDI) canli tespitinden geliyor - hic insan etiketli veriyle
karsilastirilmadi. SeaDroneSee'nin `manifest.json`'u (her klip icin gercek
tekne sayisi) Kaggle'da erisilebilirse burada YOLO'nun ne kadar isabetli
saydigini olcebiliriz.

`manifest.json` bulunamazsa (sadece video yuklediyseniz, ya da baska bir veri
kaynagi kullaniyorsaniz) bu bolum atlanir - `vehicle_count` filtresi yine
calisir, sadece dogrulugunu burada olcemeyiz.

In [ ]:
import pathlib, json
from collections import Counter
from common.qdrant_store import get_client
from common import config

_manifest_paths = (list(pathlib.Path('/kaggle/input').rglob('manifest.json'))
                    + list(pathlib.Path('/kaggle/working').rglob('manifest.json')))

if not _manifest_paths:
    print('manifest.json bulunamadi - bu bolum atlaniyor (SeaDroneSee disi veri '
          'kullaniyorsaniz beklenen davranis budur).')
else:
    manifest_path = _manifest_paths[0]
    manifest = {c['clip_id']: c.get('max_concurrent_per_category', {}).get('boat', 0)
                for c in json.loads(manifest_path.read_text(encoding='utf-8'))}
    print(f'manifest: {manifest_path} ({len(manifest)} klip)')

    client = get_client()
    points, offset = client.scroll(collection_name=config.QDRANT_COLLECTION, limit=1000,
                                    with_payload=True, with_vectors=False)
    while offset is not None:
        more, offset = client.scroll(collection_name=config.QDRANT_COLLECTION, limit=1000,
                                      offset=offset, with_payload=True, with_vectors=False)
        points += more

    yolo_max = Counter()
    for p in points:
        pl = p.payload or {}
        vid = pl.get('video_id')
        yolo_max[vid] = max(yolo_max[vid], pl.get('vehicle_count', 0) or 0)

    print(f"\n{'video':16}{'manifest(gercek)':>18}{'YOLO26(bizim)':>16}{'fark':>8}")
    errors = []
    for vid in sorted(manifest, key=lambda v: (len(v), v)):
        if vid not in yolo_max:
            continue
        m, y = manifest[vid], yolo_max[vid]
        errors.append(y - m)
        print(f"{vid:16}{m:>18}{y:>16}{y-m:>+8}")

    if errors:
        n = len(errors)
        mae = sum(abs(e) for e in errors) / n
        print(f"\nOrtalama mutlak hata: {mae:.2f} tekne, N={n}")
        print(f"Birebir dogru (fark=0): {sum(1 for e in errors if e==0)}/{n}")
    else:
        print('\nEslesen video bulunamadi (video_id isimleri manifest ile uyusmuyor olabilir).')

## 9. Sorgu testleri

vLLM olmadan yapısal ayrıştırma yok — sorgu tamamen semantiğe düşer.
Yapısal/gevşetme testleri için 9. bölüm.

In [ ]:
from query.pipeline import run_query
from scripts.query_cli import render

PROMPTS = [
    'a boat moving fast on open water',
    'people swimming near the shore',
    # Zor-negatif cifti: gorsel olarak neredeyse ayni, anlamca zit.
    # IKISI DE AYNI sonucu donduruyorsa model bu ayrimi yapamiyor (§5).
    'a boat approaching the shore at sunset',
    'a boat approaching the shore at sunrise',
]

results = {}
for p in PROMPTS:
    print('=' * 70)
    print('SORGU:', p)
    results[p] = run_query(p, top_k=10)
    render(results[p])
    print()

# Zor-negatif karsilastirmasi
a, b = results[PROMPTS[2]], results[PROMPTS[3]]
top_a = [(i.video_id, round(i.t_start)) for i in a.intervals[:3]]
top_b = [(i.video_id, round(i.t_start)) for i in b.intervals[:3]]
print('=' * 70)
print('ZOR-NEGATIF (sunset vs sunrise)')
print('  sunset top-3 :', top_a)
print('  sunrise top-3:', top_b)
print('  SONUC:', 'AYNI - model bu ayrimi YAPAMIYOR' if top_a == top_b
      else 'FARKLI - model ayrimi yapabiliyor')

### Yapısal filtre + gevşetme (vLLM'siz)

`ParsedQuery`'yi elle kurarak vLLM olmadan da yapısal yolu test edebiliriz.
`vehicle_count` YOLO'dan geliyor, telemetriye bağlı değil.

> **Küçük korpusta gevşetme her zaman tetiklenir** — bu bir hata değil.
> Eşik `SEARCH_MIN_RESULTS=5`; korpusta 5'ten az pencere varsa hiçbir filtre
> yeterli sonuç bulamaz ve merdiven sonuna kadar iner. Az veriyle test
> ediyorsanız `set_env(SEARCH_MIN_RESULTS=1)` ile düşürün.

In [ ]:
from query.llm_parser import ParsedQuery, StructuredFilters
from query.hybrid_search import search
from query.interval_merge import merge_matches

for min_v in (2, 50):   # 50 = kasitli imkansiz -> gevsetme tetiklenmeli
    p = ParsedQuery(filters=StructuredFilters(min_vehicle_count=min_v),
                    semantic_text='boats on the water', raw_query='test')
    r = search(p, top_k=10)
    iv = merge_matches(r.matches)
    print(f'--- min_vehicle_count>={min_v} ---')
    print(f'  {len(r.matches)} eslesme -> {len(iv)} aralik | '
          f'gevsetildi={r.was_relaxed} dusen={r.relaxed_fields or "(yok)"}')
    print(f'  embed={r.embed_ms:.0f}ms qdrant={r.qdrant_ms:.0f}ms '
          f'merdiven={r.ladder_steps} adim')
    for i in iv[:3]:
        print(f'    {i.video_id} {i.t_start:.0f}-{i.t_end:.0f}s '
              f'skor={i.score:.3f} tam_eslesme={i.exact_filter_match}')

## 10. vLLM — yapısal ayrıştırma (isteğe bağlı, ağır)

**Bu hat hiç doğrulanmadı** — projenin en büyük doğrulanmamış parçası.

T4 16GB'de embedding modeli (~4,3 GB) + 7B-AWQ (~5 GB) birlikte sığar ama
sıkışıktır. `--gpu-memory-utilization`'ı düşük tutun.

**Colab'da yaşadığımız (`libcudart.so.13`, vLLM'in "libtorch stable ABI"
geçişinin CUDA 13'e sabit bağımlılığı) burada da çıkabilir ya da çıkmayabilir**
- Kaggle'ın temel imajındaki torch/CUDA sürümü Colab'dan farklı. Aşağıdaki
hücre aynı önlemi (geçiş öncesi `vllm==0.8.3`) uyguluyor ve kurulan sürümü
**doğruluyor** (tahmin etmiyor).

In [ ]:
import subprocess

# SURUM GECMISI (2026-07, Colab'da AYRICA Kaggle'da gercekten yasandi):
# once --torch-backend=auto tek basina denendi (libcudart.so.13), sonra
# vllm==0.8.3'e sabitlendi (torch==2.6.0'a bagimli, o surum PyPI'dan
# KALDIRILMIS - Kaggle'da "undefined symbol: ...parseSchemaOrName..." ABI
# hatasi verdi). PyPI JSON API ile dogrulandi: v0.20.0'dan itibaren TUM
# vLLM surumleri torch==2.11.0'a sabit ve bu surum hala kurulabilir. Eski
# surume sabitlemek yerine GUNCEL vLLM'i ihtiyaci olan torch surumuyle
# birlikte ACIKCA istiyoruz.
!pip install -q uv
!uv pip install -q --system "torch==2.11.0" vllm xgrammar --torch-backend=auto 2>&1 | tail -20

# DOGRULAMA - tahmin etmiyoruz, GORUYORUZ: torch GERCEKTEN 2.11.0 mu?
result = subprocess.run(['python3', '-c',
    'import torch, vllm; print(f"TORCH: {torch.__version__}"); print(f"VLLM: {vllm.__version__}")'],
    capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('import edilemedi:', result.stderr[-500:])
elif '2.11.0' not in result.stdout.split('VLLM')[0]:
    print('*** UYARI: torch 2.11.0 DEGIL - sabitleme etkisiz kaldi. ***')
    print('Cekirdegi yeniden baslatip bu hucreyi tekrar calistirin.')

In [ ]:
import time
from common.llm import health_check

# Cikti DEVNULL'a DEGIL dosyaya: vLLM sessizce olurse (VRAM yetmezse ya da
# CUDA/kutuphane uyumsuzlugu en sik iki sebep) nedenini gorebilmeliyiz.
LOG = '/kaggle/working/vllm.log'
vllm_log = open(LOG, 'w')
import subprocess as sp
vllm_proc = sp.Popen([
    'vllm', 'serve', 'Qwen/Qwen2.5-7B-Instruct-AWQ',
    '--structured-outputs-config.backend', 'xgrammar',
    '--gpu-memory-utilization', '0.45',
    '--max-model-len', '2048',
    '--dtype', 'half',   # T4 (compute 7.5) bfloat16 desteklemiyor, half=fp16
], stdout=vllm_log, stderr=sp.STDOUT)
print(f'vLLM baslatiliyor (pid={vllm_proc.pid}) - birkac dakika surer')
print(f'Log: {LOG}   ->  !tail -30 {LOG}')

for i in range(60):
    if vllm_proc.poll() is not None:
        log_tail = open(LOG, encoding='utf-8', errors='replace').read()[-3000:]
        print(f'\nvLLM SUREC OLDU (cikis kodu {vllm_proc.returncode}). Log sonu:')
        print(log_tail)
        if 'libcudart.so' in log_tail or 'undefined symbol' in log_tail:
            print('\n*** torch==2.11.0 sabitlemesine ragmen CUDA/ABI hatasi devam ediyor. ***')
            print('Bu, vLLM #43435 ("not planned") sorununun torch versiyonundan')
            print('BAGIMSIZ, gercekten cozulmemis bir CUDA13 runtime eksikligi oldugunu')
            print('gosterir. Deneyebilecekleriniz:')
            print('  1. nvidia-smi CUDA surumunu kontrol edip elle eslesen backend verin:')
            print('       !uv pip install -q --system "torch==2.11.0" vllm --torch-backend=cu124')
            print('  2. Hedef makineniz (4060, Linux+Docker) icin docker-compose.yml')
            print('     "gpu" profili farkli bir yol - kendi CUDA runtime\'ini tasiyan')
            print('     Docker imaji bu sinif soruna hic girmeyebilir.')
            print('  Sonucu paylasin, requirements-serving.txt buna gore guncellenmeli.')
        break
    if health_check():
        print(f'vLLM hazir ({i*10}s)')
        break
    time.sleep(10)
else:
    print('vLLM 10 dakikada acilmadi. Log sonu:')
    !tail -25 {LOG}
    print('\nEn sik sebep: VRAM yetmiyor. --gpu-memory-utilization dusurun '
          'ya da embedding modelini bosaltip tekrar deneyin.')

In [ ]:
from query.pipeline import run_query
from scripts.query_cli import render

STRUCTURAL_PROMPTS = [
    'en az 3 tekne gorunen kayitlar',
    'gece deniz uzerinde hareket eden tekne',
    '50 metreden yuksekte ucan arac',
    '20 tekne olan goruntuler',        # imkansiz -> gevsetme tetiklenmeli
    'a boat on the water',             # tamamen semantik -> filtre CIKMAMALI
]

for p in STRUCTURAL_PROMPTS:
    print('=' * 70)
    print('SORGU:', p)
    render(run_query(p))
    print()

print('=' * 70)
print('BAKILACAK:')
print('  1. "Yapisal filtre:" satiri dogru alanlari mi doldurdu?')
print('     - "gece" -> is_night=True olmali')
print('     - "3 tekne" -> min_vehicle_count=3 olmali')
print('     - son sorgu (tamamen semantik) -> "filtre yok" olmali.')
print('  2. "gecikme: parse=...ms" -> vLLM ayristirmanin GERCEK maliyeti.')
print('     vLLM kapaliyken bu deger ~15ms idi (sadece geri cekilme yolu).')

## 11. Sonuçları kaydedin

Denemeden sonra şunları not edin — proje-ozeti.md §8 bunları bekliyor:

1. **Embedding gerçek-zaman katı** (batch değeriyle birlikte) — §8'in 40x
   varsayımı ne kadar uzak?
2. **`parse=...ms`** — vLLM yapısal ayrıştırmanın gerçek gecikmesi.
3. **Zor-negatif sonucu** — "sunset" ve "sunrise" farklı sonuç verdi mi?
4. **Yapısal ayrıştırma doğruluğu** — LLM alanları doğru mu dolduruyor?
5. **vLLM Kaggle'da kurulabildi mi** — hangi sürüm işe yaradı, hangi hata
   çıktıysa `requirements-serving.txt`'e işlenmeli.

Haftalık ~30 saatlik Kaggle kotanızı takip edin - sağ panelde kalan süre
gösteriliyor.

> Gecikme rakamlarını Qdrant **gömülü** modda ölçtüyseniz not düşün: o mod
> tam arama yapıyor, gerçek HNSW değil.